In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 14.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Embedding, BatchNormalization, Multiply
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (precision_score, recall_score,f1_score, accuracy_score,mean_squared_error,mean_absolute_error)
from sklearn.compose import ColumnTransformer, make_column_selector
import optuna
import tensorflow as tf
from keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import json
import os
import pickle

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(message)s'
)

In [ ]:
RAND = 1337
np.random.seed(RAND)

In [ ]:
DATA_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__"in locals() else "./"

In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR, 'UNSW_NB15_training-set.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'UNSW_NB15_testing-set.csv'))

In [ ]:
logging.info("Data Preprocessing...")

In [ ]:
train.head()

,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.000011,udp,-,INT,2,0,496,0,90909.0902,...,1,2,0,0,0,1,2,0,Normal,0
1,2,0.000008,udp,-,INT,2,0,1762,0,125000.0003,...,1,2,0,0,0,1,2,0,Normal,0
2,3,0.000005,udp,-,INT,2,0,1068,0,200000.0051,...,1,3,0,0,0,1,3,0,Normal,0
3,4,0.000006,udp,-,INT,2,0,900,0,166666.6608,...,1,3,0,0,0,2,3,0,Normal,0
4,5,0.000010,udp,-,INT,2,0,2126,0,100000.0025,...,1,3,0,0,0,2,3,0,Normal,0


In [ ]:
cols_to_drop = ["id", "attack_cat"]
train = train.drop(columns=cols_to_drop, axis=1)
test = test.drop(columns=cols_to_drop, axis=1)

In [ ]:
X_train_val_df = train.drop("label", axis=1)
y_train_val_df = train["label"]
y_test_df = test["label"]
X_test_df = test.drop("label", axis=1)

In [ ]:
X_train_df, X_val_df, y_train_df, y_val_df = train_test_split(
    X_train_val_df, y_train_val_df, test_size=0.2, random_state=RAND)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'),
         make_column_selector(dtype_include=['object', 'category'])),

        ('num', StandardScaler(),
         make_column_selector(dtype_include=['number']))
    ])

with open(os.path.join(DATA_DIR, 'models', 'preprocessor_v2.pkl'), 'wb') as f:
    pickle.dump(preprocessor, f)

FileNotFoundError: [Errno 2] No such file or directory: './models/preprocessor_v1.pkl'

In [ ]:
X_train = preprocessor.fit_transform(X_train_df)
X_val = preprocessor.transform(X_val_df)
X_test = preprocessor.transform(X_test_df)


In [ ]:
X_train = X_train.toarray()
X_val = X_val.toarray()
X_test = X_test.toarray()
y_train = np.array(y_train_df)
y_val = np.array(y_val_df)
y_test = np.array(y_test_df)

In [ ]:
print("Train Infs:", np.isinf(X_train).sum())
print("Val Infs:", np.isinf(X_val).sum())
print("Test Infs:",  np.isinf(X_test).sum())
print("Max:", X_train.max(), "Min:", X_train.min())
print("Label values:", np.unique(y_train))

Train Infs: 0
Val Infs: 0
Test Infs: 0
Max: 222.42784467165882 Min: -1.777189182579921
Label values: [0 1]


In [ ]:
logging.info(f"X_train shape: {X_train.shape}")

In [ ]:
BATCH_SIZE=1024
EPOCHS=1000

In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))
logging.info(f"Class weights: {class_weight_dict}")

In [ ]:
logging.info("Starting hyperparameter optimization")

In [ ]:
strategy = tf.distribute.MirroredStrategy()

def objective(trial):
  with strategy.scope():
    inputs = tf.keras.Input(shape=(X_train.shape[1],))

    x = Dense(1024, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)

    x = Dense(768, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)

    x = Dense(512, activation='relu')(x)
    x = Dropout(0.2)(x)

    # Camada de atenção
    attention = Dense(512, activation='softmax')(x)
    x = Multiply()([x, attention])

    output = Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(inputs=inputs, outputs=output)

    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)

    model.compile(loss='binary_crossentropy',optimizer=Adam(learning_rate=lr),metrics=['accuracy'])

  early_stop = EarlyStopping(
      monitor='val_loss',
      patience=10,
      restore_best_weights=True
  )


  model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=100,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    class_weight=class_weight_dict,
    verbose=0
  )


  loss, accuracy = model.evaluate(X_val, y_val, verbose=0)

  return accuracy

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

[I 2026-06-03 00:42:43,517] A new study created in memory with name: no-name-82ca83ed-971d-4b7d-9727-e2260dab15cb
[I 2026-06-03 00:43:23,997] Trial 0 finished with value: 0.8312382102012634 and parameters: {'lr': 0.04932538270569739}. Best is trial 0 with value: 0.8312382102012634.
[I 2026-06-03 00:44:32,155] Trial 1 finished with value: 0.966417670249939 and parameters: {'lr': 0.001925013073350809}. Best is trial 1 with value: 0.966417670249939.
[I 2026-06-03 00:46:34,457] Trial 2 finished with value: 0.969089686870575 and parameters: {'lr': 0.00018515579142001432}. Best is trial 2 with value: 0.969089686870575.
[I 2026-06-03 00:48:34,879] Trial 3 finished with value: 0.9697576761245728 and parameters: {'lr': 0.00025861597254039784}. Best is trial 3 with value: 0.9697576761245728.
[I 2026-06-03 00:49:17,127] Trial 4 finished with value: 0.9600412845611572 and parameters: {'lr': 0.007902178444173003}. Best is trial 3 with value: 0.9697576761245728.
[I 2026-06-03 00:51:18,682] Trial 5 f

In [ ]:
logging.info("Best hyperparameters:")
for key, value in study.best_params.items():
  logging.info(f"{key}: {value}")

In [ ]:
best_lr = study.best_params['lr']

In [ ]:
checkpoint = ModelCheckpoint(
    os.path.join(DATA_DIR, 'checkpoints', 'checkpoint.keras'),
    monitor='val_loss',
    save_best_only=True
)

In [ ]:
from keras.losses import BinaryCrossentropy
from keras.optimizers import Adam

In [ ]:
with strategy.scope():
  inputs = tf.keras.Input(shape=(X_train.shape[1],))

  x = Dense(1024, activation='relu')(inputs)
  x = BatchNormalization()(x)
  x = Dropout(0.2)(x)

  x = Dense(768, activation='relu')(x)
  x = BatchNormalization()(x)
  x = Dropout(0.2)(x)

  x = Dense(512, activation='relu')(x)
  x = Dropout(0.2)(x)

  # Camada de atenção
  attention = Dense(512, activation='softmax')(x)
  x = Multiply()([x, attention])

  output = Dense(1, activation='sigmoid')(x)

  model = tf.keras.Model(inputs=inputs, outputs=output)

  model.summary()

  model.compile(loss='binary_crossentropy',optimizer=Adam(learning_rate=best_lr),metrics=['accuracy'])


In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [ ]:
model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=100,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    class_weight=class_weight_dict,
    verbose=0
)
model.save(os.path.join(DATA_DIR, 'models', 'model_new_unsw_nb15_v2.keras'))

In [ ]:
y_pred = (model.predict(X_test) > 0.5).astype(int).flatten()

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred , average="binary")
precision = precision_score(y_test, y_pred , average="binary")
f1 = f1_score(y_test, y_pred, average="binary")
print("----------------------------------------------")
print("accuracy")
print("%.3f" %accuracy)
print("recall")
print("%.3f" %recall)
print("precision")
print("%.3f" %precision)
print("f1score")
print("%.3f" %f1)

In [ ]:
results = {
    "accuracy": float(accuracy),
    "recall": float(recall),
    "precision": float(precision),
    "f1": float(f1)
}
with open(os.path.join(DATA_DIR, 'models', 'results_model_new_unsw_nb15-v2.json'), 'w') as f:
    json.dump(results, f, indent=2)